# Lab 02.3 — Invoke MCP with Bearer Token

## Overview

Hora da verdade: we will invocar uma MCP tool no Gateway usando o JWT da Ana.
Sequência:

1. Login programático (igual ao Lab 01.2) → bearer token
2. Conectar ao Gateway via MCP streamable HTTP
3. Listar tools disponíveis
4. Invocar `gridapi___get_grid_status` e ver a resposta

> 💡 **Sem Cedar ainda!** Neste lab o Gateway só valida o JWT — qualquer
> authenticated user can call any tool. No Lab 03 we will add
> Cedar para diferenciar `operators` vs `managers`.

## Prerequisites

- ✅ Labs 02.1 e 02.2

## Setup

In [ ]:
import os
import sys
import json
sys.path.insert(0, "..")

from shared.utils.config import load_config, get_region
from utils import get_mcp_endpoint

cfg = load_config()
region = get_region()
mcp_url = get_mcp_endpoint(cfg["GATEWAY_URL"])
print(f"MCP endpoint: {mcp_url}")

## Step 1: Login da Ana

In [ ]:
# Importa do utils.py do Lab 01-Identity
import importlib.util
spec = importlib.util.spec_from_file_location("identity_utils", "../01-Identity-Foundation/utils.py")
identity_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(identity_utils)

ana_tokens = identity_utils.get_bearer_token(
    pool_id=cfg["COGNITO_USER_POOL_ID"],
    client_id=cfg["COGNITO_CLIENT_ID"],
    username="ana.operadora@workshop.local",
    password="Workshop@2025!",
    region=region,
)
ana_token = ana_tokens["access_token"]
print(f"Token: {ana_token[:60]}...")

## Step 2: Conectar via MCP client e listar tools

Usamos o pacote `mcp` (Model Context Protocol) com transport `streamable-http`.
O JWT vai como bearer token no header Authorization.

In [ ]:
# pip install mcp já is em requirements.txt
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def list_tools():
    headers = {"Authorization": f"Bearer {ana_token}"}
    async with streamablehttp_client(mcp_url, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            return tools

tools = await list_tools()
print(f"Tools disponíveis: {len(tools.tools)}\n")
for t in tools.tools:
    print(f"  • {t.name}")

## Step 3: Invocar uma tool — `gridapi___get_grid_status`

We request the status of sector `leste` (which has a voltage alert in the mock data).

In [ ]:
async def call_tool():
    headers = {"Authorization": f"Bearer {ana_token}"}
    async with streamablehttp_client(mcp_url, headers=headers) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(
                name="gridapi___get_grid_status",
                arguments={"sector": "leste"},
            )
            return result

result = await call_tool()
for content in result.content:
    print(content.text)

## Step 4: Tentar sem token — deve falhar

Validate that the Gateway actually requires authentication.

In [ ]:
async def call_without_token():
    async with streamablehttp_client(mcp_url, headers={}) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await session.list_tools()

try:
    await call_without_token()
    print("❌ ERRO — deveria ter rejeitado!")
except Exception as e:
    print(f"✓ Esperado: rejeitou sem token → {type(e).__name__}: {str(e)[:200]}")

## ✅ Validation

- ✓ Listou tools com token de Ana
- ✓ Chamou `get_grid_status` e recebeu dados
- ✓ Rejeitou requisição sem token

## 🎓 What you learned

- Como conectar a um Gateway MCP via streamable HTTP
- JWT vai no header `Authorization: Bearer <token>`
- Sem Cedar, any authenticated user has full access

## Next

➡️ [Lab 03 — AgentCore Policy](../03-AgentCore-Policy/)

We will adicionar Cedar policies que diferenciam operators vs managers.